In [ ]:
import sklearn
import xgboost

import src.features
import src.preprocessing
import src.training
import src.evaluation

FEATURES  = list(src.features.FEATURES_CC1E0PI.keys()) # this is a dictionary with binnings
TARGET    = src.features.TARGET_CC1E0PI
MORE_VARS = src.features.MORE_VARS_CC1E0PI

In [ ]:
DATA_PATH = "../data"
MODEL_PATH = "../models"
MODEL_NAME = "cc1e0pi_tagger_xgb"

In [ ]:
# this is the 'CV': it's the nominal NuMI flux,
# so we'll have around 5-6% of nues, which means
# something like 2-3% of the signal class (1eNp0π)
df_nom = src.preprocessing.import_cafana_data(
    f"{DATA_PATH}/NuECC0PiTagger_Features_NuMI_Nom.txt",
    FEATURES + TARGET + MORE_VARS,
    1,
    delimiter = "\t",
    index_col = False
)

# I've found that augmenting the target class
# is not really helpful -- it's best to keep it
# realistic and just pay attention to the imbalance
df_fullosc = src.preprocessing.import_cafana_data(
    f"{DATA_PATH}/NuECC0PiTagger_Features_NuMI_FullOsc.txt",
    FEATURES + TARGET + MORE_VARS,
    0,
    delimiter = "\t",
    index_col = False
)

In [ ]:
X_train, X_test, Y_train, Y_test = src.preprocessing.make_dataset(
    df_nom,
    FEATURES,
    'slice_is_cc1e0pi',
    test_size = 0.15
)

#### Train a XGBoost model with stratified k-CV grid-search

In [ ]:
# do you want to dump the training dataset to pickle?
# this will be useful for interpretation...
SAVE_TRAINING_DATASET = False

if SAVE_TRAINING_DATASET:
    X_train.to_pickle(f'{DATA_PATH}/training_dataset_X.pkl')
    Y_train.to_pickle(f'{DATA_PATH}/training_dataset_Y.pkl')

In [ ]:
# this parameter grid has been already optimized...
param_grid = {
    "n_estimators"     : [400],
    "max_depth"        : [3],
    "learning_rate"    : [0.02],
    "subsample"        : [0.7],
    "colsample_bytree" : [0.5],
    "min_child_weight" : [5],
    "gamma"            : [0.01],
    "reg_alpha"        : [0],
    "reg_lambda"       : [6]
}

model, df_grid_results = src.training.grid_search_cv_xgb(
    X_train,
    Y_train,
    param_grid = param_grid,
    n_cv_splits = 5,
)

In [ ]:
fig, ax = src.training.plot_cv_results(df_grid_results)

In [ ]:
# save the model
model.save_model(f"{MODEL_PATH}/{MODEL_NAME}.json")

#### Evaluate performance

In [ ]:
USE_LOADED_MODEL = True

if USE_LOADED_MODEL:
    model = xgboost.XGBClassifier()
    model.load_model(f"{MODEL_PATH}/{MODEL_NAME}.json")

In [ ]:
# get model predictions
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

df_test = src.evaluation.make_test_df(df_nom, X_test, model)

In [ ]:
# get main metrics
auc = sklearn.metrics.roc_auc_score(Y_test, preds)
recall = sklearn.metrics.recall_score(Y_test, preds)
precision = sklearn.metrics.precision_score(Y_test, preds)
precs, recs, thrs = sklearn.metrics.precision_recall_curve(Y_test, probs)

In [ ]:
fig, ax = src.evaluation.plot_precision_recall_threshold(thrs, precs, recs, 0.35)

In [ ]:
fig, ax = src.evaluation.plot_score_distribution(
    df_test,
    categories = src.features.CC1E0PI_CATEGORIES,
    xlabel     = "1eNp0π score",
)